# Lab 4: Bandit challenge

You write one agent, which plays three bandit problems (the tracks) without knowing in
advance which one it faces. The agent follows the interface of Lab 3: `act(features)`
returns the arm to play, given the array of the feature vectors of the arms, and
`update(x, arm, reward)` receives the result. All rewards are Bernoulli.

The score of a track is the mean final pseudo-regret of your agent divided by the one of
UCB1, which ignores the features, on the same instances, in percent: UCB1 scores $100$, and
the lower the better. The score of the challenge is the mean of the three track scores.

This version runs without installation, for instance on Google Colab (File > Upload
notebook). The cells of the section "Framework" below write the course package `seqlearn`
next to the notebook: run them first, and again after each reconnection, since Colab erases
the files at the end of a session.

## Framework

The cells of this section write the course package `seqlearn` next to the notebook. Run them, without modifying them, before the rest of the notebook.

In [ ]:
import os

os.makedirs("seqlearn", exist_ok=True)

In [ ]:
%%writefile seqlearn/bandits.py
"""Stochastic bandits: environments, agent interface, interaction loop."""

import numpy as np


class Bandit:
    """Stochastic K-armed bandit with arm means `means`."""

    def __init__(self, means, rng=None):
        self.means = np.asarray(means, dtype=float)
        self.n_arms = len(self.means)
        self.rng = np.random.default_rng(rng)

    @property
    def best_mean(self):
        return self.means.max()

    @property
    def gaps(self):
        return self.best_mean - self.means

    def pull(self, arm):
        """Pull arm `arm` and return the reward."""
        raise NotImplementedError


class BernoulliBandit(Bandit):
    def pull(self, arm):
        return float(self.rng.random() < self.means[arm])


class GaussianBandit(Bandit):
    def __init__(self, means, sigma=1.0, rng=None):
        super().__init__(means, rng)
        self.sigma = sigma

    def pull(self, arm):
        return self.means[arm] + self.sigma * self.rng.standard_normal()


class Agent:
    """Bandit agent. Subclasses implement `act`.

    `update` keeps track of `t` (number of rounds played), `counts[a]` (number of pulls
    of arm a) and `mu_hat[a]` (empirical mean of arm a, zero until the arm is pulled),
    using the incremental update seen in the lecture.
    """

    def __init__(self, n_arms, rng=None):
        self.n_arms = n_arms
        self.rng = np.random.default_rng(rng)
        self.reset()

    def reset(self):
        self.t = 0
        self.counts = np.zeros(self.n_arms, dtype=int)
        self.mu_hat = np.zeros(self.n_arms)

    def act(self):
        """Return the arm to play at the next round."""
        raise NotImplementedError

    def update(self, arm, reward):
        self.t += 1
        self.counts[arm] += 1
        self.mu_hat[arm] += (reward - self.mu_hat[arm]) / self.counts[arm]


class UniformAgent(Agent):
    def act(self):
        return int(self.rng.integers(self.n_arms))


def run(env, agent, horizon):
    """Let `agent` play on `env` for `horizon` rounds.

    Return two arrays of length `horizon`: arms played and rewards.
    """
    agent.reset()
    arms = np.empty(horizon, dtype=int)
    rewards = np.empty(horizon)
    for t in range(horizon):
        arm = agent.act()
        if not (isinstance(arm, (int, np.integer)) and 0 <= arm < env.n_arms):
            raise ValueError(
                f"act() must return an integer between 0 and {env.n_arms - 1}, "
                f"but returned {arm!r} at round {t + 1}."
            )
        reward = env.pull(arm)
        agent.update(arm, reward)
        arms[t], rewards[t] = arm, reward
    return arms, rewards


def run_many(make_env, make_agent, horizon, n_runs, seed=0):
    """Repeat `run` over `n_runs` independent and reproducible runs.

    `make_env(rng)` builds the environment and `make_agent(n_arms, rng)` the agent. Each
    run gets its own generators, derived from `seed`: two calls with the same seed see the
    same environment randomness, whatever the agent.
    Return two arrays of shape (n_runs, horizon): arms played and rewards.
    """
    arms = np.empty((n_runs, horizon), dtype=int)
    rewards = np.empty((n_runs, horizon))
    for i, child in enumerate(np.random.SeedSequence(seed).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(np.random.default_rng(env_seed))
        agent = make_agent(env.n_arms, np.random.default_rng(agent_seed))
        arms[i], rewards[i] = run(env, agent, horizon)
    return arms, rewards


def pseudo_regret(arms, gaps):
    """Cumulative pseudo-regret, sum of gaps[A_s] for s <= t, along the last axis."""
    return np.cumsum(np.asarray(gaps)[arms], axis=-1)

In [ ]:
%%writefile seqlearn/agents.py
"""Reference agents from the previous labs, used as baselines."""

import numpy as np

from .bandits import Agent


class UCBAgent(Agent):
    """UCB(alpha): index mu_hat + sqrt(alpha log t / N). alpha = 2 gives UCB1 (Labs 1 and 2)."""

    def __init__(self, n_arms, alpha=2.0, rng=None):
        self.alpha = alpha
        super().__init__(n_arms, rng)

    def index(self):
        return self.mu_hat + np.sqrt(self.alpha * np.log(self.t + 1) / self.counts)

    def act(self):
        if self.t < self.n_arms:
            return self.t
        return int(np.argmax(self.index()))

In [ ]:
%%writefile seqlearn/linear.py
"""Linear bandits: environments, agent interface, interaction loop."""

import numpy as np


class LinearBandit:
    """Linear bandit with Bernoulli rewards.

    At each round, arm a has a feature vector x_a in R^d, given by `features()` as an
    array of shape (n_arms, dim). Pulling arm a pays 1 with probability <theta, x_a> and 0
    otherwise. The parameter `theta` is unknown to the agent.
    """

    def __init__(self, theta, rng=None):
        self.theta = np.asarray(theta, dtype=float)
        self.dim = len(self.theta)
        self.rng = np.random.default_rng(rng)

    def features(self):
        """Feature vectors of the arms at the current round, of shape (n_arms, dim)."""
        raise NotImplementedError

    def means(self):
        """Mean rewards of the arms at the current round."""
        return self.features() @ self.theta

    def pull(self, arm):
        """Pull arm `arm`, return the reward and move to the next round."""
        reward = float(self.rng.random() < self.means()[arm])
        self._next_round()
        return reward

    def _next_round(self):
        pass


class CatalogBandit(LinearBandit):
    """Fixed catalog: item a has the feature vector items[a], the same at every round."""

    def __init__(self, items, theta, rng=None):
        super().__init__(theta, rng)
        self.items = np.asarray(items, dtype=float)
        self.n_arms = len(self.items)

    def features(self):
        return self.items


class RecommendationBandit(LinearBandit):
    """Recommendation with user contexts.

    `appeal[a, j]` is the probability that item a is clicked by a user in mood j. At each
    round a user arrives with a profile `context`, a probability vector over the moods, and
    item a is clicked with probability <context, appeal[a]>. This is a linear bandit of
    dimension n_arms * n_moods: the feature vector of item a is the profile written in
    block a, and zero elsewhere, and theta is the matrix `appeal` flattened.
    """

    def __init__(self, appeal, concentration=0.3, rng=None):
        self.appeal = np.asarray(appeal, dtype=float)
        self.n_arms, self.n_moods = self.appeal.shape
        self.concentration = concentration
        super().__init__(self.appeal.ravel(), rng)
        self._next_round()

    def _next_round(self):
        self.context = self.rng.dirichlet(self.concentration * np.ones(self.n_moods))
        self._features = np.kron(np.eye(self.n_arms), self.context)

    def features(self):
        return self._features


def make_catalog(n_items, dim, rng=None):
    """Random catalog: `n_items` unit feature vectors with nonnegative entries, and a unit
    parameter theta with nonnegative entries, so that all mean rewards lie in [0, 1]."""
    rng = np.random.default_rng(rng)
    items = np.abs(rng.standard_normal((n_items, dim)))
    items /= np.linalg.norm(items, axis=1, keepdims=True)
    theta = np.abs(rng.standard_normal(dim))
    theta /= np.linalg.norm(theta)
    return items, theta


class LinearAgent:
    """Linear bandit agent. Subclasses implement `act` and `update`."""

    def __init__(self, dim, rng=None):
        self.dim = dim
        self.rng = np.random.default_rng(rng)
        self.reset()

    def reset(self):
        self.t = 0

    def act(self, features):
        """Return the arm to play, given the array `features` of shape (n_arms, dim)."""
        raise NotImplementedError

    def update(self, x, arm, reward):
        """Take into account the reward of arm `arm`, whose feature vector is `x`."""
        self.t += 1


class ContextFree(LinearAgent):
    """Wrap a K-armed agent (Labs 1 and 2) so that it plays a linear bandit, ignoring the
    features: each arm index is treated as an unrelated arm."""

    def __init__(self, agent):
        self.agent = agent
        super().__init__(dim=None, rng=agent.rng)

    def reset(self):
        super().reset()
        self.agent.reset()

    def act(self, features):
        return self.agent.act()

    def update(self, x, arm, reward):
        super().update(x, arm, reward)
        self.agent.update(arm, reward)


def run_linear(env, agent, horizon):
    """Let `agent` play on the linear bandit `env` for `horizon` rounds.

    Return three arrays of length `horizon`: arms played, rewards, and instantaneous
    regrets max_a <theta, x_a> - <theta, x_{A_t}>.
    """
    agent.reset()
    arms = np.empty(horizon, dtype=int)
    rewards = np.empty(horizon)
    regrets = np.empty(horizon)
    for t in range(horizon):
        features = env.features()
        arm = agent.act(features)
        if not (isinstance(arm, (int, np.integer)) and 0 <= arm < len(features)):
            raise ValueError(
                f"act() must return an integer between 0 and {len(features) - 1}, "
                f"but returned {arm!r} at round {t + 1}."
            )
        means = features @ env.theta
        reward = env.pull(arm)
        agent.update(features[arm], arm, reward)
        arms[t], rewards[t], regrets[t] = arm, reward, means.max() - means[arm]
    return arms, rewards, regrets


def run_many_linear(make_env, make_agent, horizon, n_runs, seed=0):
    """Repeat `run_linear` over `n_runs` independent and reproducible runs.

    `make_env(rng)` builds the environment and `make_agent(env, rng)` the agent.
    Return the cumulative pseudo-regret, of shape (n_runs, horizon).
    """
    regret = np.empty((n_runs, horizon))
    for i, child in enumerate(np.random.SeedSequence(seed).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(np.random.default_rng(env_seed))
        agent = make_agent(env, np.random.default_rng(agent_seed))
        regret[i] = np.cumsum(run_linear(env, agent, horizon)[2])
    return regret

In [ ]:
%%writefile seqlearn/plotting.py
"""Shared plotting: course palette and mean curves with uncertainty bands."""

from math import lgamma

import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

# Categorical palette, to be used in this order (validated for color vision deficiencies on
# adjacent pairs). Beyond 4 series, prefer several subplots.
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TEXT = "#52514e"
MUTED = "#a3a29c"
GRID = "#e6e5e1"


def set_style():
    plt.rcParams.update({
        "axes.prop_cycle": cycler(color=COLORS),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": MUTED,
        "axes.labelcolor": TEXT,
        "axes.titlecolor": TEXT,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": TEXT,
        "ytick.color": TEXT,
        "lines.linewidth": 2,
        "lines.solid_capstyle": "round",
        "legend.frameon": False,
        "legend.labelcolor": TEXT,
        "font.size": 10,
        "figure.dpi": 110,
        "savefig.bbox": "tight",
    })


def plot_mean(ax, curves, label=None, band="se", color=None, end_label=False, label_dy=0, **kwargs):
    """Plot the mean of `curves`, of shape (n_runs, horizon), with an uncertainty band.

    band="se": mean +/- 2 standard errors; band="quantile": 10% and 90% quantiles.
    end_label=True writes `label` at the end of the curve, in addition to the legend,
    shifted vertically by `label_dy` points to avoid overlaps.
    """
    curves = np.asarray(curves, dtype=float)
    t = np.arange(1, curves.shape[1] + 1)
    mean = curves.mean(axis=0)
    if band == "se":
        half = 2 * curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
        low, high = mean - half, mean + half
    elif band == "quantile":
        low, high = np.quantile(curves, [0.1, 0.9], axis=0)
    else:
        raise ValueError("band must be 'se' or 'quantile'.")
    (line,) = ax.plot(t, mean, label=label, color=color, **kwargs)
    ax.fill_between(t, low, high, color=line.get_color(), alpha=0.15, linewidth=0)
    if end_label and label:
        ax.annotate(label, xy=(t[-1], mean[-1]), xytext=(6, label_dy), textcoords="offset points",
                    va="center", color=TEXT, fontsize="small")
    return line


def beta_pdf(x, a, b):
    """Density of the Beta(a, b) distribution at the points `x` of [0, 1]."""
    x = np.clip(np.asarray(x, dtype=float), 1e-12, 1 - 1e-12)
    log_norm = lgamma(a + b) - lgamma(a) - lgamma(b)
    return np.exp(log_norm + (a - 1) * np.log(x) + (b - 1) * np.log(1 - x))

In [ ]:
%%writefile seqlearn/checks.py
"""Automatic tests for the labs. Each function raises AssertionError with an explicit message."""

import numpy as np

from .bandits import Bandit, run


class _FixedBandit(Bandit):
    """Deterministic bandit: arm a always pays means[a]."""

    def pull(self, arm):
        return float(self.means[arm])


class _DecayBandit(Bandit):
    """Arm 0 pays 1 at its first pull and 0 afterwards; arm 1 always pays 0.5."""

    def __init__(self):
        super().__init__([0.0, 0.5])
        self._first = True

    def pull(self, arm):
        if arm == 1:
            return 0.5
        reward, self._first = float(self._first), False
        return reward


def _arms(agent, env, horizon):
    arms, _ = run(env, agent, horizon)
    return arms.tolist()


def _ok(name):
    print(f"{name}: all tests passed.")


def check_pseudo_regret(pseudo_regret):
    gaps = np.array([0.0, 0.1, 0.3])
    arms = np.array([0, 1, 1, 2])
    expected = np.array([0.0, 0.1, 0.2, 0.5])
    out = pseudo_regret(arms, gaps)
    assert out is not None, "pseudo_regret returns nothing."
    assert np.allclose(out, expected), (
        f"For arms={arms.tolist()} and gaps={gaps.tolist()}, "
        f"expected {expected.tolist()}, got {np.asarray(out).tolist()}."
    )
    batch = np.stack([arms, arms[::-1]])
    out = pseudo_regret(batch, gaps)
    assert np.shape(out) == (2, 4), "For arms of shape (n_runs, T), the output must have the same shape."
    assert np.allclose(out, [expected, [0.3, 0.4, 0.5, 0.5]]), (
        "The cumulative sum must run along the last axis (time)."
    )
    _ok("pseudo_regret")


def check_greedy(GreedyAgent):
    got = _arms(GreedyAgent(3), _FixedBandit([0.2, 0.8, 0.5]), 6)
    assert got == [0, 1, 2, 1, 1, 1], (
        f"Noiseless means (0.2, 0.8, 0.5): expected [0, 1, 2, 1, 1, 1], got {got}. "
        "Each arm must be pulled once, in order, before taking the argmax."
    )
    got = _arms(GreedyAgent(2), _FixedBandit([0.5, 0.5]), 4)
    assert got == [0, 1, 0, 0], (
        f"Equal means: expected [0, 1, 0, 0], got {got}. Ties must go to the smallest index."
    )
    agent = GreedyAgent(2)
    _arms(agent, _FixedBandit([0.9, 0.1]), 5)
    got = _arms(agent, _FixedBandit([0.1, 0.9]), 4)
    assert got == [0, 1, 1, 1], "reset() must bring the agent back to its initial state."
    _ok("GreedyAgent")


def check_epsilon_greedy(EpsilonGreedyAgent):
    got = _arms(EpsilonGreedyAgent(3, 0.0), _FixedBandit([0.2, 0.8, 0.5]), 6)
    assert got == [0, 1, 2, 1, 1, 1], f"With epsilon = 0, the agent must behave like greedy: got {got}."
    horizon = 4002
    arms = np.array(_arms(EpsilonGreedyAgent(3, 1.0, rng=0), _FixedBandit([0.2, 0.8, 0.5]), horizon))
    freqs = np.bincount(arms[3:], minlength=3) / (horizon - 3)
    assert np.allclose(freqs, 1 / 3, atol=0.04), (
        f"With epsilon = 1, arms must be uniform: frequencies {freqs.round(3)}."
    )
    arms = np.array(_arms(EpsilonGreedyAgent(2, 0.3, rng=0), _FixedBandit([0.0, 1.0]), horizon))
    freq = np.mean(arms[2:] == 0)
    assert abs(freq - 0.15) < 0.03, (
        "With epsilon = 0.3 and two arms, the bad arm must be played about 15% of the time "
        f"(uniform exploration, hence epsilon / 2): got {100 * freq:.1f}%."
    )
    _ok("EpsilonGreedyAgent")


def check_etc(ETCAgent):
    got = _arms(ETCAgent(3, 2), _FixedBandit([0.3, 0.7, 0.5]), 9)
    assert got == [0, 1, 2, 0, 1, 2, 1, 1, 1], (
        f"ETC(m=2) on (0.3, 0.7, 0.5): expected [0, 1, 2, 0, 1, 2, 1, 1, 1], got {got}. "
        "During exploration, arms are played in turn."
    )
    got = _arms(ETCAgent(2, 3), _FixedBandit([0.5, 0.5]), 8)
    assert got == [0, 1, 0, 1, 0, 1, 0, 0], (
        f"Equal means: expected [0, 1, 0, 1, 0, 1, 0, 0], got {got}. Ties must go to the smallest index."
    )
    got = _arms(ETCAgent(2, 1), _DecayBandit(), 6)
    assert got == [0, 1, 0, 0, 0, 0], (
        f"Expected [0, 1, 0, 0, 0, 0], got {got}. After exploration, ETC plays the same arm "
        "until the end, even if the empirical means change."
    )
    _ok("ETCAgent")


# ---------------------------------------------------------------------------
# Lab 2

def check_ucb(UCBAgent):
    agent = UCBAgent(2, alpha=2.0)
    for arm, reward in [(0, 1.0), (1, 0.0), (0, 0.0)]:
        agent.update(arm, reward)
    out = agent.index()
    assert out is not None, "index returns nothing."
    assert np.shape(out) == (2,), f"index must return an array of length n_arms: got shape {np.shape(out)}."
    expected = np.array([0.5, 0.0]) + np.sqrt(2.0 * np.log(4) / np.array([2, 1]))
    shifted = np.array([0.5, 0.0]) + np.sqrt(2.0 * np.log(3) / np.array([2, 1]))
    assert not np.allclose(out, shifted), (
        "After 3 rounds, the index is the one of round t = 4: the logarithm is log(self.t + 1), "
        "not log(self.t)."
    )
    assert not np.allclose(out, [0.5, 0.0]), "The index is the empirical mean alone: the bonus is missing."
    assert np.allclose(out, expected), (
        f"After the pulls (arm 0, reward 1), (arm 1, reward 0), (arm 0, reward 0), with alpha = 2, "
        f"expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    agent = UCBAgent(2, alpha=0.5)
    for arm, reward in [(0, 1.0), (1, 0.0), (0, 0.0)]:
        agent.update(arm, reward)
    expected = np.array([0.5, 0.0]) + np.sqrt(0.5 * np.log(4) / np.array([2, 1]))
    assert np.allclose(agent.index(), expected), (
        f"With alpha = 0.5, expected {expected.round(3).tolist()}, "
        f"got {np.asarray(agent.index()).round(3).tolist()}. The bonus must depend on self.alpha."
    )
    got = _arms(UCBAgent(3, alpha=2.0), _FixedBandit([0.2, 0.8, 0.5]), 3)
    assert got == [0, 1, 2], f"During the first K rounds, each arm must be pulled once, in order: got {got}."
    got = _arms(UCBAgent(3, alpha=2.0), _FixedBandit([0.2, 0.8, 0.5]), 12)
    expected = [0, 1, 2, 1, 2, 1, 0, 1, 2, 1, 1, 2]
    assert got == expected, (
        f"Noiseless means (0.2, 0.8, 0.5), alpha = 2: expected {expected}, got {got}. "
        "After the first K rounds, play the argmax of the index."
    )
    got = _arms(UCBAgent(2, alpha=0.5), _FixedBandit([0.9, 0.1]), 14)
    expected = [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    assert got == expected, f"Noiseless means (0.9, 0.1), alpha = 0.5: expected {expected}, got {got}."
    agent = UCBAgent(2, alpha=2.0)
    _arms(agent, _FixedBandit([0.9, 0.1]), 5)
    got = _arms(agent, _FixedBandit([0.1, 0.9]), 4)
    assert got == [0, 1, 1, 1], "reset() must bring the agent back to its initial state."
    _ok("UCBAgent")


# ---------------------------------------------------------------------------
# Lab 3

def _feed(agent, history):
    for arm, reward in history:
        agent.update(arm, reward)


def check_thompson(ThompsonAgent):
    agent = ThompsonAgent(2, rng=0)
    _feed(agent, [(0, 1.0), (0, 1.0), (0, 0.0), (1, 0.0)])
    out = agent.posterior()
    assert out is not None and len(out) == 2, "posterior must return the pair (alpha, beta)."
    alpha, beta = np.asarray(out[0], dtype=float), np.asarray(out[1], dtype=float)
    assert alpha.shape == (2,) and beta.shape == (2,), "alpha and beta must be arrays of length n_arms."
    assert np.allclose(alpha, [3, 1]) and np.allclose(beta, [2, 2]), (
        "After 2 successes and 1 failure on arm 0, and 1 failure on arm 1, with uniform priors, "
        f"expected alpha = [3, 1] and beta = [2, 2], got alpha = {alpha.tolist()} and beta = {beta.tolist()}."
    )
    agent = ThompsonAgent(3, rng=0)
    draws = np.array([agent.act() for _ in range(3000)])
    assert np.all(np.isin(draws, [0, 1, 2])), "act() must return an arm index."
    freqs = np.bincount(draws, minlength=3) / len(draws)
    assert np.allclose(freqs, 1 / 3, atol=0.04), (
        f"Before any observation, the three arms must be played with the same probability: frequencies {freqs.round(3)}. "
        "Thompson sampling has no initialization rounds."
    )
    agent = ThompsonAgent(2, rng=0)
    _feed(agent, [(0, 1.0)] * 60 + [(1, 0.0)] * 60)
    draws = [agent.act() for _ in range(200)]
    assert all(arm == 0 for arm in draws), "After 60 successes on arm 0 and 60 failures on arm 1, arm 0 must always be played."
    agent = ThompsonAgent(2, rng=0)
    _feed(agent, [(0, 1.0), (1, 0.0)])
    freq = np.mean([agent.act() == 0 for _ in range(6000)])
    assert abs(freq - 5 / 6) < 0.02, (
        "After one success on arm 0 and one failure on arm 1, arm 0 must be played with probability 5/6 "
        f"(Tutorial 3, exercise 1): got {freq:.3f}."
    )
    first, second = ThompsonAgent(2, rng=7), ThompsonAgent(2, rng=7)
    assert [first.act() for _ in range(50)] == [second.act() for _ in range(50)], (
        "Two agents with the same seed must make the same choices: use self.rng for all random draws."
    )
    _ok("ThompsonAgent")


def _ridge_data():
    rng = np.random.default_rng(0)
    xs = rng.standard_normal((6, 3))
    rs = rng.random(6)
    lam = 2.0
    V = lam * np.eye(3) + xs.T @ xs
    b = xs.T @ rs
    return xs, rs, lam, V, b


def check_ridge(Ridge):
    xs, rs, lam, V, b = _ridge_data()
    ridge = Ridge(3, lam)
    assert np.allclose(ridge.V_inv, np.eye(3) / lam), "Before any observation, V_inv must be the identity divided by lam."
    assert np.allclose(ridge.theta_hat, 0), "Before any observation, theta_hat must be zero."
    for x, r in zip(xs, rs):
        ridge.update(x, r)
    assert np.allclose(ridge.V_inv, np.linalg.inv(V)), (
        "After 6 updates, V_inv differs from the inverse of lam * I + sum of x x^T: check the Sherman-Morrison update."
    )
    assert np.allclose(ridge.b, b), "After 6 updates, b must be the sum of the r x."
    assert np.allclose(ridge.theta_hat, np.linalg.solve(V, b)), "theta_hat must be V_inv @ b, recomputed after each update."
    X = np.array([[1.0, 0.0, 0.0], [0.0, 2.0, 0.0], [1.0, 1.0, 1.0], [0.5, -1.0, 2.0]])
    out = ridge.width(X)
    assert out is not None, "width returns nothing."
    expected = np.sqrt(np.sum(X @ np.linalg.inv(V) * X, axis=1))
    assert np.shape(out) == (4,), f"For features of shape (4, 3), width must return an array of length 4: got shape {np.shape(out)}."
    assert np.allclose(out, expected), (
        f"width must return sqrt(x^T V_inv x) for each row x: expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    _ok("Ridge")


_LIN_HISTORY = [([1.0, 0.0], 1.0), ([1.0, 0.0], 1.0), ([1.0, 0.0], 0.0), ([1.0, 0.0], 0.0), ([0.0, 1.0], 0.0)]
_LIN_FEATURES = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])


def _feed_linear(agent):
    for x, reward in _LIN_HISTORY:
        agent.update(np.array(x), 0, reward)


def check_linucb(LinUCBAgent):
    agent = LinUCBAgent(2, beta=1.0, lam=1.0)
    out = agent.index(np.array([[1.0, 0.0], [2.0, 0.0]]))
    assert out is not None, "index returns nothing."
    assert np.allclose(out, [1.0, 2.0]), (
        f"Before any observation, with beta = 1 and lam = 1, the index of x is its norm: expected [1, 2], got {np.asarray(out).round(3).tolist()}."
    )
    _feed_linear(agent)
    out = np.asarray(agent.index(_LIN_FEATURES))
    expected = np.array([0.4 + np.sqrt(0.2), np.sqrt(0.5), 0.4 + np.sqrt(0.7)])
    assert np.allclose(out, expected), (
        "Example of Tutorial 3 (exercise 3): V = diag(5, 2), theta_hat = (0.4, 0), beta = 1. "
        f"Expected indices {expected.round(3).tolist()}, got {out.round(3).tolist()}."
    )
    assert agent.act(_LIN_FEATURES) == 2, "On the example of Tutorial 3, LinUCB must play the third arm."
    assert isinstance(agent.act(_LIN_FEATURES), (int, np.integer)), "act() must return an integer."
    agent = LinUCBAgent(2, beta=0.0, lam=1.0)
    _feed_linear(agent)
    assert agent.act(_LIN_FEATURES) == 0, "With beta = 0, the agent plays the argmax of <theta_hat, x>, ties going to the smallest index."
    agent.reset()
    assert np.allclose(agent.index(np.array([[1.0, 0.0]])), 0.0), "reset() must forget the observations."
    _ok("LinUCBAgent")


def check_lints(LinTSAgent):
    agent = LinTSAgent(2, v=0.5, lam=1.0, rng=0)
    _feed_linear(agent)
    samples = np.array([agent.sample() for _ in range(6000)])
    assert samples.shape == (6000, 2), "sample() must return a vector of length dim."
    mean, cov = samples.mean(axis=0), np.cov(samples.T)
    assert np.allclose(mean, [0.4, 0.0], atol=0.02), (
        f"The samples must be centered at theta_hat = (0.4, 0): empirical mean {mean.round(3).tolist()}."
    )
    expected = 0.25 * np.diag([0.2, 0.5])
    assert np.allclose(cov, expected, atol=0.01), (
        "The covariance of the samples must be v^2 V_inv = diag(0.05, 0.125): "
        f"empirical covariance {cov.round(3).tolist()}."
    )
    agent = LinTSAgent(2, v=0.0, lam=1.0, rng=0)
    _feed_linear(agent)
    assert all(agent.act(_LIN_FEATURES[:2]) == 0 for _ in range(20)), "With v = 0, the agent plays the argmax of <theta_hat, x>."
    first, second = LinTSAgent(2, v=1.0, lam=1.0, rng=3), LinTSAgent(2, v=1.0, lam=1.0, rng=3)
    assert [first.act(_LIN_FEATURES) for _ in range(30)] == [second.act(_LIN_FEATURES) for _ in range(30)], (
        "Two agents with the same seed must make the same choices: use self.rng for all random draws."
    )
    _ok("LinTSAgent")


# ---------------------------------------------------------------------------
# Lab 0

def check_gaps(gaps):
    out = gaps(np.array([0.5, 0.6, 0.4]))
    assert out is not None, "gaps returns nothing."
    assert np.allclose(out, [0.1, 0.0, 0.2]), (
        f"For means (0.5, 0.6, 0.4), expected [0.1, 0.0, 0.2], got {np.asarray(out).tolist()}."
    )
    _ok("gaps")


def check_counts(counts):
    out = counts(np.array([0, 2, 2, 0, 2]), 4)
    assert out is not None, "counts returns nothing."
    assert np.shape(out) == (4,), (
        f"With n_arms = 4, the output must have length 4, even if an arm is never played: got shape {np.shape(out)}."
    )
    assert np.array_equal(out, [2, 0, 3, 0]), (
        f"For arms [0, 2, 2, 0, 2] and n_arms = 4, expected [2, 0, 3, 0], got {np.asarray(out).tolist()}."
    )
    _ok("counts")


def check_running_mean(running_mean):
    x = np.array([1.0, 0.0, 1.0, 1.0])
    expected = np.array([1.0, 0.5, 2 / 3, 0.75])
    out = running_mean(x)
    assert out is not None, "running_mean returns nothing."
    assert np.allclose(out, expected), (
        f"For x = {x.tolist()}, expected {expected.round(3).tolist()}, got {np.asarray(out).round(3).tolist()}."
    )
    batch = np.stack([x, 1 - x])
    out = running_mean(batch)
    assert np.shape(out) == (2, 4), "For x of shape (n_runs, T), the output must have the same shape."
    assert np.allclose(out, [expected, 1 - expected]), "The running mean must be computed along the last axis (time)."
    _ok("running_mean")


def check_bernoulli(bernoulli):
    out = bernoulli(0.3, 20000, np.random.default_rng(0))
    assert out is not None, "bernoulli returns nothing."
    out = np.asarray(out)
    assert out.shape == (20000,), f"With size = 20000, the output must have shape (20000,): got {out.shape}."
    assert np.all((out == 0) | (out == 1)), "The values must be 0 or 1."
    assert abs(out.mean() - 0.3) < 0.02, f"With p = 0.3, the mean of the draws must be close to 0.3: got {out.mean():.3f}."
    again = np.asarray(bernoulli(0.3, 20000, np.random.default_rng(0)))
    assert np.array_equal(out, again), "Two calls with generators of the same seed must return the same draws: use `rng` for all random draws."
    out = np.asarray(bernoulli(0.5, (3, 7), np.random.default_rng(1)))
    assert out.shape == (3, 7), f"With size = (3, 7), the output must have shape (3, 7): got {out.shape}."
    _ok("bernoulli")


def check_running_mean_class(RunningMean):
    m = RunningMean()
    assert m.n == 0, "A new RunningMean must have n = 0."
    values = []
    for x in [1.0, 0.0, 1.0, 1.0]:
        m.update(x)
        values.append(m.value)
    assert m.n == 4, f"After 4 updates, n must be 4: got {m.n}."
    assert np.allclose(values, [1.0, 0.5, 2 / 3, 0.75]), (
        f"After the updates 1, 0, 1, 1, the successive values must be [1, 0.5, 0.667, 0.75]: got {np.round(values, 3).tolist()}."
    )
    stored = [v for v in vars(m).values() if isinstance(v, (list, np.ndarray)) and np.size(v) > 1]
    assert not stored, "The past observations must not be stored: keep only n and value."
    _ok("RunningMean")


def check_round_robin(RoundRobinAgent):
    got = _arms(RoundRobinAgent(3), _FixedBandit([0.2, 0.8, 0.5]), 7)
    assert got == [0, 1, 2, 0, 1, 2, 0], f"With 3 arms, expected [0, 1, 2, 0, 1, 2, 0], got {got}."
    _ok("RoundRobinAgent")

In [ ]:
%%writefile seqlearn/challenge.py
"""Bandit challenge (Lab 4): three tracks, one agent, a score relative to UCB1.

An agent is built by a function `make_agent(n_arms, dim, horizon, rng)` and follows the
interface of `LinearAgent`: `act(features)` returns the arm to play and
`update(x, arm, reward)` receives the result. The three tracks are linear bandits with
Bernoulli rewards:

  slots    K = 20 arms with small means (click rates), features e_1, ..., e_K
  catalog  K = 300 items described by d = 8 features, the same at every round
  users    K = 6 items and a user profile over 4 moods at each round, d = 24

The score of a track is the mean final pseudo-regret of the agent divided by the one of
UCB1, which ignores the features, on the same instances, in percent. The score of the
challenge is the mean of the three track scores: the lower, the better.
"""

import importlib.util
import time
from pathlib import Path

import numpy as np

from .agents import UCBAgent
from .linear import CatalogBandit, ContextFree, RecommendationBandit, make_catalog, run_linear

HORIZON = 5000
TRACKS = ("slots", "catalog", "users")


def make_env(track, rng):
    """Draw an instance of `track` (its parameters and its randomness) from `rng`."""
    rng = np.random.default_rng(rng)
    if track == "slots":
        n_arms = 20
        return CatalogBandit(np.eye(n_arms), rng.beta(1.5, 12, size=n_arms), rng)
    if track == "catalog":
        items, theta = make_catalog(300, 8, rng)
        return CatalogBandit(items, theta, rng)
    if track == "users":
        return RecommendationBandit(rng.beta(0.7, 2.0, size=(6, 4)), concentration=0.3, rng=rng)
    raise ValueError(f"unknown track {track!r}, expected one of {TRACKS}")


def make_reference(n_arms, dim, horizon, rng):
    """UCB1 on unrelated arms, the reference of the scores."""
    return ContextFree(UCBAgent(n_arms, 2.0, rng))


def final_regrets(make_agent, track, n_runs, seed):
    """Final pseudo-regret of `n_runs` runs on `track`, on instances derived from `seed`."""
    regrets = np.empty(n_runs)
    for i, child in enumerate(np.random.SeedSequence([seed, TRACKS.index(track)]).spawn(n_runs)):
        env_seed, agent_seed = child.spawn(2)
        env = make_env(track, np.random.default_rng(env_seed))
        dim = env.features().shape[1]
        agent = make_agent(env.n_arms, dim, HORIZON, np.random.default_rng(agent_seed))
        regrets[i] = run_linear(env, agent, HORIZON)[2].sum()
    return regrets


def evaluate(make_agent, n_runs=10, seed=0, verbose=True):
    """Evaluate an agent on the three tracks, against the reference.

    Return a dictionary: for each track, the mean final regret of the agent and of the
    reference, and the score; and the overall score under the key "score".
    """
    results = {}
    for track in TRACKS:
        start = time.perf_counter()
        agent = final_regrets(make_agent, track, n_runs, seed)
        elapsed = time.perf_counter() - start
        reference = final_regrets(make_reference, track, n_runs, seed)
        score = 100 * agent.mean() / reference.mean()
        results[track] = {"regret": agent.mean(), "reference": reference.mean(), "score": score,
                          "seconds": elapsed}
        if verbose:
            print(f"{track:8s} regret {agent.mean():7.1f} (UCB1 {reference.mean():6.1f})"
                  f"  score {score:5.1f}  [{elapsed:.0f} s]")
    results["score"] = float(np.mean([results[track]["score"] for track in TRACKS]))
    if verbose:
        print(f"score of the challenge: {results['score']:.1f} (UCB1: 100, lower is better)")
    return results


def load_agent(path):
    """Load the function `make_agent` of the file `path`, reloading the file at each call."""
    path = Path(path)
    spec = importlib.util.spec_from_file_location(f"challenge_{path.stem}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.make_agent

In [ ]:
%%writefile seqlearn/__init__.py
"""Lab framework for the Sequential Decision course (ESIEE Paris)."""

from .bandits import (
    Agent,
    Bandit,
    BernoulliBandit,
    GaussianBandit,
    UniformAgent,
    pseudo_regret,
    run,
    run_many,
)
from .agents import UCBAgent
from .plotting import plot_mean

In [ ]:
import numpy as np

from seqlearn.challenge import HORIZON, TRACKS, evaluate, load_agent, make_env

## 1. The tracks

Each track is a distribution over instances: `make_env(track, rng)` draws one. The horizon
is $T = 5000$ on every track.

- `slots`: $K = 20$ unrelated arms with small means, like click rates. The feature vector
  of arm $a$ is $e_a$: this is a $K$-armed bandit written as a linear bandit (Tutorial 3,
  exercise 3, question 1).
- `catalog`: $K = 300$ items described by $d = 8$ features, the same at every round, as
  the catalog of Lab 3.
- `users`: $K = 6$ items. At each round a new user arrives, with a profile over $4$ moods,
  and the feature vectors of the items depend on this profile ($d = 24$). The best item
  changes from one round to the next.

In [ ]:
for track in TRACKS:
    env = make_env(track, 0)
    features = env.features()
    means = features @ env.theta
    print(f"{track:8s} features {features.shape}, mean rewards: best {means.max():.3f}, "
          f"median {np.median(means):.3f}")

env = make_env("users", 0)
for t in range(3):
    print(f"users, round {t + 1}: best item {np.argmax(env.means())}, means {env.means().round(2)}")
    env.pull(0)

## 2. The reference

An agent is a file that defines a function `make_agent(n_arms, dim, horizon, rng)`. The
cell below writes the file of the reference, UCB1 on unrelated arms, with the tools of
`seqlearn`. Then `load_agent` reads the file again and `evaluate` plays $10$ runs on each
track. By construction, the reference scores $100$ on each track.

In [ ]:
%%writefile agent_ucb1.py
from seqlearn.agents import UCBAgent
from seqlearn.linear import ContextFree


def make_agent(n_arms, dim, horizon, rng):
    return ContextFree(UCBAgent(n_arms, 2.0, rng))

In [ ]:
evaluate(load_agent("agent_ucb1.py"));

## 3. Your agent

Rename the file in the first line of the cell below, and in the call to `load_agent` of
the next cell, to `agent_` followed by your names, for instance `agent_dupont_martin.py`,
and write your agent. The cell must contain
everything the agent needs: the imports, the classes (you may copy those of Lab 3) and
the function `make_agent`.

A few leads, from the simplest to the most ambitious:

- the agents of Lab 3 play the three tracks without any change;
- the parameters $\beta$ and $v$ were tuned on the catalog of Lab 3, not on these tracks;
- on `slots`, the means are small: the uniform prior of Thompson sampling, or the bonus of
  UCB1, which is designed for rewards in $[0, 1]$, may be far from the right scale;
- `make_agent` receives `n_arms` and `dim`, and the feature vectors tell the agent which
  structure it faces.

In [ ]:
%%writefile agent_yourname.py
import numpy as np

from seqlearn.linear import LinearAgent


class MyAgent(LinearAgent):
    def __init__(self, dim, rng=None):
        super().__init__(dim, rng)

    def reset(self):
        super().reset()
        ...  # TODO

    def act(self, features):
        ...  # TODO

    def update(self, x, arm, reward):
        super().update(x, arm, reward)
        ...  # TODO


def make_agent(n_arms, dim, horizon, rng):
    return MyAgent(dim, rng)

In [ ]:
evaluate(load_agent("agent_yourname.py"));

## 4. Rules

Your agent, in its file `agent_....py`, must follow these rules:

1. the file defines `make_agent(n_arms, dim, horizon, rng)` and only imports `numpy` and
   `seqlearn`;
2. all the randomness of the agent comes from `rng`;
3. `evaluate` with its default parameters runs in less than $3$ minutes on your laptop;
4. the agent only learns from `features`, `x`, `arm` and `reward`: it has no access to
   the environment.